# 5G Network Intelligence — Temporal Residual MLP

## Hypothesis and prior evidence

The linear history model regressed at every tested horizon/fold. Test a small nonlinear residual network while keeping the same input policy, training budget and temporal windows. Compare one-KPI A and four-KPI B; no gain is assumed. A fit-only spatial diagnostic found weak average same-cell residual association, so graph complexity is not added in this study.

## Temporal protocol

Fit/stopping/scoring cutoffs: 335/383/443 and 395/443/503. Stopping selects epochs, scoring cannot tune architecture. All original beams and complete per-horizon scoring origins are retained. These development folds were reused and cannot prove fresh-test generalization. Validation 504–671 and final holdout are excluded.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = 'da1e8401a71ec50ed8f3bd899d84f5b4191afa41'


## Configuration

Four hosted GPU fits: A/B on two folds, one seed. All beams participate in fitting; no beam identity or summary is a numerical model input.


In [ ]:
PROTOCOL = {'protocol': 'temporal-residual-mlp-development-v1',
 'variants': ['A', 'B'],
 'seed': 17,
 'horizons': [1, 6, 24],
 'folds': [{'name': 'inner1', 'fit_end': 335, 'stop_end': 383, 'score_end': 443},
           {'name': 'inner2', 'fit_end': 395, 'stop_end': 443, 'score_end': 503}],
 'hidden_size': 64,
 'dropout': 0.1,
 'learning_rate': 0.001,
 'weight_decay': 0.01,
 'batch_size': 1024,
 'max_epochs': 30,
 'early_stopping_patience': 5,
 'gradient_clip': 1.0,
 'training_stride': 1,
 'beam_population': 'all_original_beams',
 'preprocessing': 'global_per_KPI_fit_prefix_only_with_missing_masks',
 'initialization': 'causal_target_seasonal_mean_7d_plus_zero_residual_head',
 'phase_inputs': 4,
 'validation_504_671': 'forbidden',
 'original_final_holdout': 'forbidden',
 'production_change': 'forbidden'}


## Preprocessing and architecture

Fit global per-KPI medians/means/std on the training prefix only. Preserve missing masks and phase4. Center physical history by its last imputed/scaled value; a 64-unit GELU network with dropout 0.1 predicts a nonlinear residual. Add the causal seven-day seasonal target mean. A zero-initialized final head starts from that analytical baseline. All weights are shared across beams; no fitted beam summary or neighbouring beam values enter. Full source: `src/models/temporal_mlp.py`.


## Hosted source and canonical inputs

Attach the completed original hurdle development output for shared code and canonical panel, plus the small MIT temporal-MLP source dataset. Only the new model, trainer and protocol are copied from that source attachment.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots = [p.parent.parent for p in Path('/kaggle/input').rglob('train_hurdle_research.py') if (p.parent.parent/'data/processed/panel.npz').exists()]
    if len(roots) != 1:
        raise RuntimeError('Attach exactly one canonical source output')
    PROJECT = Path('/kaggle/working/temporal_mlp')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src','scripts','configs','data/processed'):
        shutil.copytree(roots[0]/folder,PROJECT/folder,dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0,str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = SOURCE_COMMIT
    Path('reports').mkdir(exist_ok=True)
    patches = list(Path('/kaggle/input').rglob('train_temporal_mlp.py'))
    if len(patches) != 1:
        raise RuntimeError('Attach exactly one temporal-MLP source')
    patch = patches[0].parent
    for filename,folder in (('train_temporal_mlp.py','scripts'),('temporal_mlp.py','src/models'),('temporal_mlp.yaml','configs')):
        shutil.copyfile(patch/filename,Path(folder)/filename)


## GPU verification

Use the hosted image's PyTorch installation. A missing GPU stops fitting; there is no local or CPU training fallback.


In [ ]:
if RUN_TRAINING:
    import torch
    print('PyTorch:',torch.__version__)
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPU is required')
    subprocess.run(['nvidia-smi'],check=True)


## Training and debugging

The shared masked-L1 trainer writes best/last checkpoint, optimizer/RNG state and per-epoch history. AdamW weight decay is fixed at its 0.01 default. The stopping score is joint scaled L1; scoring reports original-unit MAE after nonnegative clipping. Failures preserve partial artifacts and a bounded progress log.


In [ ]:
if RUN_TRAINING:
    with Path('reports/temporal_mlp_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','-m','scripts.train_temporal_mlp'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    with tarfile.open('/kaggle/working/temporal_mlp_results.tar.gz','w:gz') as archive:
        for folder in ('reports','artifacts/temporal_mlp'):
            if Path(folder).exists():
                archive.add(folder,arcname=folder)
    if status:
        raise RuntimeError('Training failed; inspect saved log and checkpoints')
else:
    print('Review mode: no fitting, backward or optimizer steps.')


## Scoring, negative results and limitations

Every horizon has its own complete scoring origins, so short horizons are not restricted by +24h label availability. Report MAE in original units, WAPE as a ratio, coverage and zero false-positive rate. One seed and reused development folds cannot establish generalization. No production promotion or holdout replay.


In [ ]:
paths = [Path('reports/temporal_mlp_development.csv'),Path('../reports/temporal_mlp_development.csv')]
if Path('/kaggle/input').is_dir():
    paths += list(Path('/kaggle/input').rglob('temporal_mlp_development.csv'))
path = next((p for p in paths if p.exists()),None)
if path:
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(path))
else:
    print('No completed scores attached; no improvement is assumed.')
